In [ ]:
import pandas as pd
import numpy as np
import random
#import faker
#namegen = faker.Faker()
from Handler import *

pygame 2.6.1 (SDL 2.28.4, Python 3.13.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
0


c:\Users\leojo\OneDrive\Documents\GitHub\AWH_Curling3\League.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  FADF = pd.concat([ROOKIEDF, EXPIREDF]).sort_values(['Rating', 'Age'], ascending=[False, True])
c:\Users\leojo\OneDrive\Documents\GitHub\AWH_Curling3\League.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  NEEDSDF = TEAMSDF.merge(RosterDF.groupby(['Team'], sort = False).agg(Spent = ('AAV', 'sum'), NR = ('AAV', 'count')), how = 'left', left_index=True, right_index=True).fillna(0)
c:\Users\leojo\OneDrive\Documen

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
FA DEMO
Rating      86.70
Age             7
Country    Brazil
League     Brazil
Name: Debra Tate, dtype: object
                       Country  LeaguePrestige  TeamPrestige  PlayersNeeded  \
London          United Kingdom             100            50           1.00   
Belo Horizonte          Brazil              70            50           2.00   
Salvador                Brazil              70            50           2.00   
Recife                  Brazil              70            50           1.00   
Brasilia                Brazil              70            50           1.00   
Guarulhos               Brazil              70            50           1.00   
Manaus                  Brazil              70            50           1.00   
Goiania                 Brazil              70            50           1.00   
Belem                   Brazil              70            50           1.00   
Porto Alegre            Brazil              70    

In [27]:
            
class Bracket:
    def __init__(self, name, teams, format = 'default'):
        self.name = name
        self.teams = teams
        self.format = format
        self.losers = []
        if format == 'default':
            self.blist = list(teams)
        elif format == 'CC':
            self.blist = [None]*(CCBracketPlaces[len(teams)-1]+1)
            for i in range(len(teams)):
                self.blist[CCBracketPlaces[i]] = teams[i]
        elif format == '2-sided':
            self.blist = list(teams)
        self.ogblist = self.blist
        self.depth = int(np.ceil(np.log2(len(self.blist))))
        self.winner = None

    def pygameDisplay(self):
        bracket(self.blist, self.ogblist)

    def displayNextSlate(self):
        diffTarget = 2**self.depth+1
        for i in range(diffTarget//2):
            if self.blist[i] is not None:
                tarind = diffTarget - i - 2
                if len(self.blist) > tarind and self.blist[tarind] is not None:
                    print(f'#{index_ignore_none(self.blist, self.blist[i])+1} {self.blist[i]} vs #{index_ignore_none(self.blist, self.blist[tarind])+1} {self.blist[tarind]}')

    def playNextSlate(self):
        if self.depth == 0:
            print("Bracket complete.")
            return
        diffTarget = 2**self.depth+1
        for i in range(diffTarget//2):
            if self.blist[i] is not None:
                tarind = diffTarget - i - 2
                if len(self.blist) > tarind and self.blist[tarind] is not None:
                    result = game_LoRes(self.blist[i], self.blist[tarind], comp = f'{self.name} Bracket Depth {self.depth}')
                    winner = result[0] if result[1] > result[2] else result[3]
                    self.blist[i] = winner
                    self.blist[tarind] = None
        self.depth -= 1
        self.blist = trim_trailing_none(self.blist)
        if self.depth == 0:
            self.winner = self.blist[0]

In [ ]:
class LEAGUE:
    def __init__(self, Name, Teams):
        self.name = Name
        self.slate = 0
        self.STANDINGS = pd.DataFrame(index = Teams, columns = ['Wins', 'Losses', 'EEWins', 'PD'], data=0)
        self.SCHEDULE = round_robin(Teams)
        self.BRACKET = None
        self.tWinner = None
        self.prequal = []

    def amIDone(self):
        return self.slate > 22
    
    def playNext(self):
        if self.slate < 19:
            for match in self.SCHEDULE[self.slate]:
                Result = game_LoRes(match[0], match[1])
                self.STANDINGS.loc[Result[0], 'PD'] += Result[1]-Result[2]
                self.STANDINGS.loc[Result[3], 'PD'] += Result[2]-Result[1]
                if Result[1] > Result[2]:
                    self.STANDINGS.loc[Result[0], 'Wins'] += 1
                    if Result[5]:
                        self.STANDINGS.loc[Result[0], 'EEWins'] += 1
                    self.STANDINGS.loc[Result[3], 'Losses'] += 1
                else:
                    self.STANDINGS.loc[Result[3], 'Wins'] += 1
                    if Result[5]:
                        self.STANDINGS.loc[Result[3], 'EEWins'] += 1
                    self.STANDINGS.loc[Result[0], 'Losses'] += 1
            self.STANDINGS = self.STANDINGS.sort_values(['Wins', 'EEWins', 'PD'], ascending=[False, True, False])
            if self.slate == 18:
                self.BRACKET = Bracket(self.name, list(self.STANDINGS.index[:7]) + [None, None] + list(self.STANDINGS.index[7:10]))
        elif self.slate < 23:
            self.BRACKET.playNextSlate()
            if self.slate == 22:
                self.tWinner = self.BRACKER.winner
                self.prequal.append(self.BRACKET.winner)
        self.slate += 1
        
            

                

TestLeague = LEAGUE('United States', TEAMSDF[TEAMSDF.Country == 'United States'].index)

In [ ]:
class LEAGUE:
    def __init__(self, Name, Teams):
        self.name = Name
        self.slate = 0
        self.STANDINGS = pd.DataFrame(index = Teams, columns = ['Wins', 'Losses', 'EEWins', 'PD'], data=0)
        self.SCHEDULE = round_robin(Teams)
        self.BRACKET = None
        self.tWinner = None
        self.prequal = []

    def amIDone(self):
        return self.slate > 22
    
    def playNext(self):
        if self.slate < 19:
            for match in self.SCHEDULE[self.slate]:
                Result = game_LoRes(match[0], match[1])
                self.STANDINGS.loc[Result[0], 'PD'] += Result[1]-Result[2]
                self.STANDINGS.loc[Result[3], 'PD'] += Result[2]-Result[1]
                if Result[1] > Result[2]:
                    self.STANDINGS.loc[Result[0], 'Wins'] += 1
                    if Result[5]:
                        self.STANDINGS.loc[Result[0], 'EEWins'] += 1
                    self.STANDINGS.loc[Result[3], 'Losses'] += 1
                else:
                    self.STANDINGS.loc[Result[3], 'Wins'] += 1
                    if Result[5]:
                        self.STANDINGS.loc[Result[3], 'EEWins'] += 1
                    self.STANDINGS.loc[Result[0], 'Losses'] += 1
            self.STANDINGS = self.STANDINGS.sort_values(['Wins', 'EEWins', 'PD'], ascending=[False, True, False])
            if self.slate == 18:
                self.BRACKET = Bracket(self.name, list(self.STANDINGS.index[:7]) + [None, None] + list(self.STANDINGS.index[7:10]))
        elif self.slate < 23:
            self.BRACKET.playNextSlate()
            if self.slate == 22:
                self.tWinner = self.BRACKER.winner
                self.prequal.append(self.BRACKET.winner)
        self.slate += 1

    def popQual(self):
        Next = self.STANDINGS.remove(self.prequal).index.iloc[0]
        self.prequal.append(Next)
        return Next
        
            

                

TestLeague = LEAGUE('United States', TEAMSDF[TEAMSDF.Country == 'United States'].index)

In [30]:
for _ in range(19):
    TestLeague.playNext()

TypeError: unsupported operand type(s) for -: 'NoneType' and 'NoneType'

In [34]:
TestLeague.STANDINGS

,Wins,Losses,EEWins,PD
New York,0,0,0,0
Los Angeles,0,0,0,0
Chicago,0,0,0,0
Houston,0,0,0,0
Philadelphia,0,0,0,0
Phoenix,0,0,0,0
San Antonio,0,0,0,0
San Diego,0,0,0,0
Dallas,0,0,0,0
San Jose,0,0,0,0


In [ ]:
TestLeague.preQual.append(TestLeague.STANDINGS.index[3])